# Chapter 11: Fine-tuning BERT - Hard Tasks

This notebook covers advanced fine-tuning techniques: Masked Language Modeling (MLM), Named Entity Recognition (NER), and custom entity extraction.

## Setup

Run all cells in this section to set up the environment and load the data.

Before running these cells, review the concepts from the main Chapter 11 notebook.

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [1]:
# %%capture
# !pip install "datasets>=2.18.0,<3" transformers>=4.38.2 accelerate>=0.27.2 seqeval>=1.2.2

### Import Libraries

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForTokenClassification
from transformers import DataCollatorForLanguageModeling, DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import pipeline
import numpy as np
import evaluate

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Challenges

Complete the following tasks by implementing the starter code.

### Level: Hard

**About This Task:**

Masked Language Modeling (MLM) is a self-supervised pre-training objective. The model learns to predict masked tokens, improving its understanding of language and domain-specific vocabulary.

#### Hard Task 1: Masked Language Modeling on Rotten Tomatoes

### Instructions

1. Load BERT for Masked Language Modeling
2. Prepare Rotten Tomatoes data (remove labels)
3. Create data collator with 15% masking probability
4. Train the model with MLM objective
5. Test predictions on movie-related text

Load the data.

In [3]:
# Load Rotten Tomatoes dataset
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

Load model for MLM.

In [4]:
# Load model for Masked Language Modeling (MLM)
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

Tokenize the data (no labels needed).

In [5]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

# Tokenize data and remove labels (MLM is unsupervised)
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_train = tokenized_train.remove_columns("label")
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_test = tokenized_test.remove_columns("label")

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map: 100%|██████████| 1066/1066 [00:00<00:00, 28427.46 examples/s]


View a tokenized example.

In [6]:
# Examine tokenized data
print("Tokenized example (no labels):")
print(tokenized_train[0])

Tokenized example (no labels):
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'input_ids': [101, 1103, 2067, 1110, 17348, 1106, 1129, 1103, 6880, 1432, 112, 188, 1207, 107, 14255, 1389, 107, 1105, 1115, 1119, 112, 188, 1280, 1106, 1294, 170, 24194, 1256, 3407, 1190, 170, 11791, 5253, 188, 1732, 7200, 10947, 12606, 2895, 117, 179, 7766, 118, 172, 15554, 1181, 3498, 6961, 3263, 1137, 188, 1566, 7912, 14516, 6997, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


Create data collator for masking tokens.

In [7]:
# Masking Tokens with 15% probability
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

Define training arguments.

In [8]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
    "mlm_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

Create trainer.

In [9]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

Save the tokenizer before training.

In [10]:
# Save pre-trained tokenizer
tokenizer.save_pretrained("mlm")

('mlm/tokenizer_config.json',
 'mlm/special_tokens_map.json',
 'mlm/vocab.txt',
 'mlm/added_tokens.json',
 'mlm/tokenizer.json')

Train the model.

Save the trained model.

In [11]:
# Save updated model
model.save_pretrained("mlm")

### Task 1a: Compare Before and After MLM

Test predictions on the same text with the base model and fine-tuned model.

Test with base BERT (no fine-tuning).

In [12]:
# Load and create predictions with base model
mask_filler_base = pipeline("fill-mask", model="bert-base-cased")
preds_base = mask_filler_base("What a horrible [MASK]!")

print("Base BERT predictions:")
for pred in preds_base:
    print(f">>> {pred['sequence']}")

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

Base BERT predictions:
>>> What a horrible idea!
>>> What a horrible dream!
>>> What a horrible thing!
>>> What a horrible day!
>>> What a horrible thought!


Test with fine-tuned model.

In [13]:
# Load and create predictions with fine-tuned model
mask_filler = pipeline("fill-mask", model="mlm")
preds = mask_filler("What a horrible [MASK]!")

print("\nFine-tuned BERT predictions:")
for pred in preds:
    print(f">>> {pred['sequence']}")


Fine-tuned BERT predictions:
>>> What a horrible idea!
>>> What a horrible dream!
>>> What a horrible thing!
>>> What a horrible day!
>>> What a horrible thought!


Test on more movie-related examples.

In [14]:
# Test case 2
test_text_2 = "This [MASK] is amazing!"

print(f"\nTest: {test_text_2}")
print("\nBase BERT:")
for pred in mask_filler_base(test_text_2)[:3]:
    print(f"  {pred['sequence']}")

print("\nFine-tuned:")
for pred in mask_filler(test_text_2)[:3]:
    print(f"  {pred['sequence']}")


Test: This [MASK] is amazing!

Base BERT:
  This place is amazing!
  This house is amazing!
  This man is amazing!

Fine-tuned:
  This place is amazing!
  This house is amazing!
  This man is amazing!


In [15]:
# Test case 3
test_text_3 = "The [MASK] was disappointing."

print(f"\nTest: {test_text_3}")
print("\nBase BERT:")
for pred in mask_filler_base(test_text_3)[:3]:
    print(f"  {pred['sequence']}")

print("\nFine-tuned:")
for pred in mask_filler(test_text_3)[:3]:
    print(f"  {pred['sequence']}")


Test: The [MASK] was disappointing.

Base BERT:
  The result was disappointing.
  The outcome was disappointing.
  The view was disappointing.

Fine-tuned:
  The result was disappointing.
  The outcome was disappointing.
  The view was disappointing.


### Questions

1. How do the predictions differ between base and fine-tuned models?

2. Does the fine-tuned model predict more movie-related words? Give examples.

3. Why is MLM useful for domain adaptation? (Hint: learns domain-specific vocabulary)

**About This Task:**

Named Entity Recognition (NER) extracts structured information from text by identifying entities like people, organizations, and locations. This requires token-level classification.

#### Hard Task 2: Named Entity Recognition on CoNLL-2003

### Instructions

1. Load CoNLL-2003 NER dataset
2. Understand the label schema (B-PER, I-PER, B-ORG, etc.)
3. Align labels with sub-word tokens
4. Train token classification model
5. Evaluate with sequential F1 score

Load the CoNLL-2003 dataset.

In [16]:
# The CoNLL-2003 dataset for NER
dataset = load_dataset("conll2003", trust_remote_code=True)

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 11713.90 examples/s]


Examine a training example.

In [17]:
# Examine an example
example = dataset["train"][848]
print("Example from training set:")
print(example)

Example from training set:
{'id': '848', 'tokens': ['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.'], 'pos_tags': [22, 22, 38, 29, 16, 21, 15, 12, 23, 7], 'chunk_tags': [11, 12, 21, 11, 12, 12, 13, 11, 12, 0], 'ner_tags': [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]}


View the tokens and NER tags together.

In [18]:
# Show tokens with their labels
print("\nTokens with NER tags:")
for token, tag in zip(example["tokens"], example["ner_tags"]):
    print(f"  {token:15} -> {tag}")


Tokens with NER tags:
  Dean            -> 1
  Palmer          -> 2
  hit             -> 0
  his             -> 0
  30th            -> 0
  homer           -> 0
  for             -> 0
  the             -> 0
  Rangers         -> 3
  .               -> 0


Define the label mappings.

In [19]:
# Label mappings
label2id = {
    'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4,
    'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8
}
id2label = {index: label for label, index in label2id.items()}

print("Label schema:")
print("  O: Outside any entity")
print("  B-PER: Beginning of person name")
print("  I-PER: Inside person name")
print("  B-ORG: Beginning of organization")
print("  I-ORG: Inside organization")
print("  B-LOC: Beginning of location")
print("  I-LOC: Inside location")
print("  B-MISC: Beginning of miscellaneous entity")
print("  I-MISC: Inside miscellaneous entity")

Label schema:
  O: Outside any entity
  B-PER: Beginning of person name
  I-PER: Inside person name
  B-ORG: Beginning of organization
  I-ORG: Inside organization
  B-LOC: Beginning of location
  I-LOC: Inside location
  B-MISC: Beginning of miscellaneous entity
  I-MISC: Inside miscellaneous entity


Load model for token classification.

In [20]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Understand sub-word tokenization.

In [21]:
# Split individual tokens into sub-tokens
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("\nOriginal tokens:")
print(example["tokens"])
print("\nSub-word tokens:")
print(sub_tokens)


Original tokens:
['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.']

Sub-word tokens:
['[CLS]', 'Dean', 'Palmer', 'hit', 'his', '30th', 'home', '##r', 'for', 'the', 'Rangers', '.', '[SEP]']


Define label alignment function.

In [22]:
def align_labels(examples):
    """Align NER labels with sub-word tokens"""
    token_ids = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]
    
    updated_labels = []
    for index, label in enumerate(labels):
        
        # Map tokens to their respective word
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            
            # The start of a new word
            if word_idx != previous_word_idx:
                
                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)
            
            # Special token is -100
            elif word_idx is None:
                label_ids.append(-100)
            
            # If the label is B-XXX we change it to I-XXX for sub-words
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)
        
        updated_labels.append(label_ids)
    
    token_ids["labels"] = updated_labels
    return token_ids

Apply label alignment to the dataset.

In [23]:
# Tokenize and align labels
tokenized = dataset.map(align_labels, batched=True)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 25780.43 examples/s]


Compare original and aligned labels.

In [24]:
# Difference between original and updated labels
print("\nLabel alignment example:")
print(f"Original tokens: {example['tokens']}")
print(f"Original labels: {example['ner_tags']}")
print(f"Sub-word tokens: {sub_tokens}")
print(f"Aligned labels:  {tokenized['train'][848]['labels']}")


Label alignment example:
Original tokens: ['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.']
Original labels: [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]
Sub-word tokens: ['[CLS]', 'Dean', 'Palmer', 'hit', 'his', '30th', 'home', '##r', 'for', 'the', 'Rangers', '.', '[SEP]']
Aligned labels:  [-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]


Define evaluation metric.

In [26]:
!pip install seqeval

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=2f50f5e298cbb7e9a25f16a6c38849c62eacc0089c5f1080d3dd3d2b50e1c12b
  Stored in directory: /home/danielcastillo/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


In [27]:
# Load sequential evaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """Calculate sequential F1 score for NER"""
    # Create predictions
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)
    
    true_predictions = []
    true_labels = []
    
    # Document-level iteration
    for prediction, label in zip(predictions, labels):
        
        # Token-level iteration
        for token_prediction, token_label in zip(prediction, label):
            
            # We ignore special tokens (-100)
            if token_label != -100:
                true_predictions.append([id2label[token_prediction]])
                true_labels.append([id2label[token_label]])
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {"f1": results["overall_f1"]}

Create data collator for token classification.

In [28]:
# Token-classification Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Define training arguments.

In [29]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
    "ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

Create trainer and train.

In [30]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

 57%|█████▋    | 502/878 [00:54<00:42,  8.95it/s]

{'loss': 0.2265, 'grad_norm': 0.7447206974029541, 'learning_rate': 8.610478359908885e-06, 'epoch': 0.57}


100%|██████████| 878/878 [01:41<00:00,  8.69it/s]

{'train_runtime': 101.0578, 'train_samples_per_second': 138.94, 'train_steps_per_second': 8.688, 'train_loss': 0.1639341536849941, 'epoch': 1.0}


TrainOutput(global_step=878, training_loss=0.1639341536849941, metrics={'train_runtime': 101.0578, 'train_samples_per_second': 138.94, 'train_steps_per_second': 8.688, 'train_loss': 0.1639341536849941, 'epoch': 1.0})

Evaluate the NER model.

In [31]:
# Evaluate the model on our test data
results = trainer.evaluate()
print(f"\nNER F1 Score: {results['eval_f1']:.4f}")

100%|██████████| 216/216 [00:06<00:00, 32.13it/s]


NER F1 Score: 0.9016


### Task 2a: Test NER on Custom Text

Use the trained model to extract entities from new text.

Save the model and create a pipeline.

In [32]:
# Save our fine-tuned model
trainer.save_model("ner_model")

# Run inference on the fine-tuned model
token_classifier = pipeline(
    "token-classification",
    model="ner_model",
)

Test on person names.

In [33]:
# Test case 1: Person name
text1 = "My name is Maarten."
predictions1 = token_classifier(text1)

print(f"Text: {text1}")
print("Entities found:")
for pred in predictions1:
    print(f"  {pred['word']:15} -> {pred['entity']} (score: {pred['score']:.3f})")

Text: My name is Maarten.
Entities found:
  Ma              -> B-PER (score: 0.978)
  ##arte          -> I-PER (score: 0.964)
  ##n             -> I-PER (score: 0.940)


Test on organizations and locations.

In [34]:
# Test case 2: Organization and location
text2 = "Apple Inc. is headquartered in Cupertino, California."
predictions2 = token_classifier(text2)

print(f"\nText: {text2}")
print("Entities found:")
for pred in predictions2:
    print(f"  {pred['word']:15} -> {pred['entity']} (score: {pred['score']:.3f})")


Text: Apple Inc. is headquartered in Cupertino, California.
Entities found:
  Apple           -> B-ORG (score: 0.975)
  Inc             -> I-ORG (score: 0.986)
  .               -> I-ORG (score: 0.972)
  Cup             -> B-LOC (score: 0.995)
  ##ert           -> I-LOC (score: 0.990)
  ##ino           -> I-LOC (score: 0.993)
  California      -> B-LOC (score: 0.982)


Test on multiple entities.

In [35]:
# Test case 3: Multiple entities
text3 = "Barack Obama was born in Hawaii and served as President from 2009 to 2017."
predictions3 = token_classifier(text3)

print(f"\nText: {text3}")
print("Entities found:")
for pred in predictions3:
    print(f"  {pred['word']:15} -> {pred['entity']} (score: {pred['score']:.3f})")


Text: Barack Obama was born in Hawaii and served as President from 2009 to 2017.
Entities found:
  Barack          -> B-PER (score: 0.982)
  Obama           -> I-PER (score: 0.985)
  Hawaii          -> B-LOC (score: 0.995)


### Questions

1. Why do we set sub-word labels to -100? (Hint: prevents duplicate counting)

2. What is the difference between B-PER and I-PER labels? Why do we need both?

3. How would you handle entity extraction for a new domain (e.g., medical entities)?

**About This Task:**

Real-world NER often requires custom entity types. You can create custom annotated data to extract domain-specific entities.

#### Hard Task 3: Advanced NER with Custom Entities

### Instructions

1. Create a custom NER dataset with movie-related entities
2. Define entity types: ACTOR, DIRECTOR, MOVIE, GENRE
3. Annotate sample sentences with these entities
4. Fine-tune BERT on custom entities
5. Test on movie review text

Define custom entity labels.

In [36]:
# Custom label schema for movies
custom_label2id = {
    'O': 0,
    'B-ACTOR': 1, 'I-ACTOR': 2,
    'B-DIRECTOR': 3, 'I-DIRECTOR': 4,
    'B-MOVIE': 5, 'I-MOVIE': 6,
    'B-GENRE': 7, 'I-GENRE': 8
}
custom_id2label = {idx: label for label, idx in custom_label2id.items()}

print("Custom entity types:")
print("  ACTOR: Actor names")
print("  DIRECTOR: Director names")
print("  MOVIE: Movie titles")
print("  GENRE: Movie genres")

Custom entity types:
  ACTOR: Actor names
  DIRECTOR: Director names
  MOVIE: Movie titles
  GENRE: Movie genres


Create sample annotated data.

In [37]:
# Create custom training examples
# Format: list of tokens and corresponding NER tags
custom_examples = [
    {
        "tokens": ["Tom", "Hanks", "starred", "in", "Forrest", "Gump", "."],
        "ner_tags": [1, 2, 0, 0, 5, 6, 0]  # B-ACTOR, I-ACTOR, O, O, B-MOVIE, I-MOVIE, O
    },
    {
        "tokens": ["Christopher", "Nolan", "directed", "Inception", "."],
        "ner_tags": [3, 4, 0, 5, 0]  # B-DIRECTOR, I-DIRECTOR, O, B-MOVIE, O
    },
    {
        "tokens": ["The", "Matrix", "is", "a", "science", "fiction", "film", "."],
        "ner_tags": [5, 6, 0, 0, 7, 8, 0, 0]  # B-MOVIE, I-MOVIE, O, O, B-GENRE, I-GENRE, O, O
    },
    {
        "tokens": ["Leonardo", "DiCaprio", "won", "an", "Oscar", "."],
        "ner_tags": [1, 2, 0, 0, 0, 0]  # B-ACTOR, I-ACTOR, O, O, O, O
    },
    {
        "tokens": ["Steven", "Spielberg", "directed", "many", "action", "movies", "."],
        "ner_tags": [3, 4, 0, 0, 7, 0, 0]  # B-DIRECTOR, I-DIRECTOR, O, O, B-GENRE, O, O
    }
]

print(f"Created {len(custom_examples)} annotated examples")
print("\nFirst example:")
print(f"Tokens: {custom_examples[0]['tokens']}")
print(f"Tags:   {custom_examples[0]['ner_tags']}")

Created 5 annotated examples

First example:
Tokens: ['Tom', 'Hanks', 'starred', 'in', 'Forrest', 'Gump', '.']
Tags:   [1, 2, 0, 0, 5, 6, 0]


Convert to dataset format.

In [38]:
from datasets import Dataset

# Convert to HuggingFace Dataset
custom_dataset = Dataset.from_dict({
    "tokens": [ex["tokens"] for ex in custom_examples],
    "ner_tags": [ex["ner_tags"] for ex in custom_examples]
})

print("Dataset created:")
print(custom_dataset)

Dataset created:
Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 5
})


### Task 3a: Add More Training Examples

Create additional annotated sentences to improve the model.

In [39]:
# Fill in: Add your own annotated examples
# Example format:
# {
#     "tokens": ["word1", "word2", ...],
#     "ner_tags": [tag1, tag2, ...]  # Use numbers from custom_label2id
# }

my_custom_examples = [
    # TODO: Add 3-5 more annotated examples
    # Example:
    # {
    #     "tokens": ["Meryl", "Streep", "is", "a", "great", "actress", "."],
    #     "ner_tags": [1, 2, 0, 0, 0, 0, 0]
    # },
]

if len(my_custom_examples) > 0:
    # Combine with original examples
    all_custom_examples = custom_examples + my_custom_examples
    custom_dataset = Dataset.from_dict({
        "tokens": [ex["tokens"] for ex in all_custom_examples],
        "ner_tags": [ex["ner_tags"] for ex in all_custom_examples]
    })
    print(f"\nUpdated dataset with {len(custom_dataset)} examples")

Load model for custom NER.

In [40]:
# Load tokenizer and model for custom entities
custom_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
custom_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(custom_id2label),
    id2label=custom_id2label,
    label2id=custom_label2id
)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenize and align labels.

In [41]:
def align_custom_labels(examples):
    """Align custom NER labels with sub-word tokens"""
    token_ids = custom_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]
    
    updated_labels = []
    for index, label in enumerate(labels):
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx != previous_word_idx:
                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)
            elif word_idx is None:
                label_ids.append(-100)
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:  # B- tag becomes I- tag
                    updated_label += 1
                label_ids.append(updated_label)
        
        updated_labels.append(label_ids)
    
    token_ids["labels"] = updated_labels
    return token_ids

# Apply tokenization and label alignment
tokenized_custom = custom_dataset.map(align_custom_labels, batched=True)

Map: 100%|██████████| 5/5 [00:00<00:00, 724.81 examples/s]


Train the custom NER model.

In [42]:
# Training arguments (more epochs for small dataset)
custom_training_args = TrainingArguments(
    "custom_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=20,  # More epochs due to small dataset
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

# Create trainer
custom_trainer = Trainer(
    model=custom_model,
    args=custom_training_args,
    train_dataset=tokenized_custom,
    tokenizer=custom_tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=custom_tokenizer),
)

# Train
custom_trainer.train()

100%|██████████| 60/60 [02:04<00:00,  2.08s/it]

{'train_runtime': 124.6603, 'train_samples_per_second': 0.802, 'train_steps_per_second': 0.481, 'train_loss': 0.7770758310953776, 'epoch': 20.0}


TrainOutput(global_step=60, training_loss=0.7770758310953776, metrics={'train_runtime': 124.6603, 'train_samples_per_second': 0.802, 'train_steps_per_second': 0.481, 'train_loss': 0.7770758310953776, 'epoch': 20.0})

Test the custom NER model.

In [43]:
# Save and load for inference
custom_trainer.save_model("custom_ner_model")

# Create pipeline
custom_classifier = pipeline(
    "token-classification",
    model="custom_ner_model",
)

Test on movie-related text.

In [ ]:
# Test case 1
test_text_1 = "Brad Pitt starred in Fight Club, directed by David Fincher."
predictions = custom_classifier(test_text_1)

print(f"Text: {test_text_1}")
print("Entities found:")
for pred in predictions:
    print(f"  {pred['word']:20} -> {pred['entity']:15} (score: {pred['score']:.3f})")

In [ ]:
# Test case 2
test_text_2 = "The Godfather is a classic crime drama."
predictions = custom_classifier(test_text_2)

print(f"\nText: {test_text_2}")
print("Entities found:")
for pred in predictions:
    print(f"  {pred['word']:20} -> {pred['entity']:15} (score: {pred['score']:.3f})")

### Questions

1. How accurate are the custom entity predictions with only 5-10 training examples?

2. What would you need to improve the model? (More data, better annotations, etc.)

3. How would you evaluate a custom NER model when you don't have a large test set?